<a href="https://colab.research.google.com/github/ble27/CSCE-491/blob/main/FP32_INT8_Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring FP32 to INT8 Quantization


In [ ]:
# Install Python packages without replacing Colab's CUDA-enabled PyTorch.
%pip install -q transformers "optimum[exporters]"

Thu Oct  1 22:32:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P0             30W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Shared packages for TensorRT and non-TensorRT pipeline


In [ ]:
# Shared packages for both pipelines. Do not install or upgrade torch,
# torchvision, numpy, pandas, or protobuf in Colab.
%pip install -q -U "setuptools<82" wheel jedi
%pip install -q -U \
  "onnx>=1.18.0" \
  onnxscript \
  onnxsim \
  onnxruntime \
  "optimum-onnx[onnxruntime]" \
  "nvidia-modelopt[onnx]" \
  cuda-python \
  timm \
  transformers

### Non-TensorRT download


In [ ]:
# Non-TensorRT pipeline dependencies. This uses CPU ONNX Runtime.
%pip install -q -U "optimum-onnx[onnxruntime]" onnxruntime

### TensortRT download

In [30]:
!pip install tensorrt
import tensorrt
print(tensorrt.__version__)

11.3.0.99


In [ ]:
# Runtime preflight. The ONNX Runtime pipeline can run on CPU, but ModelOpt
# and TensorRT require a real NVIDIA GPU runtime.
import shutil
import subprocess
import sys


def check_gpu_runtime():
    nvidia_smi = shutil.which("nvidia-smi")
    cuda_available = False
    cuda_version = None
    device_name = None

    try:
        import torch
        cuda_available = torch.cuda.is_available()
        cuda_version = torch.version.cuda
        if cuda_available:
            device_name = torch.cuda.get_device_name(0)
    except Exception as error:
        print(f"PyTorch CUDA check failed: {error}")

    print("Python:", sys.version.split()[0])
    print("nvidia-smi:", nvidia_smi or "not found")
    print("CUDA available:", cuda_available)
    print("CUDA version:", cuda_version)
    print("GPU:", device_name or "not available")

    if nvidia_smi:
        subprocess.run([nvidia_smi, "--query-gpu=name", "--format=csv,noheader"], check=False)

    return bool(nvidia_smi and cuda_available)


GPU_RUNTIME_AVAILABLE = check_gpu_runtime()


def require_gpu_runtime():
    if not GPU_RUNTIME_AVAILABLE:
        raise RuntimeError(
            "TensorRT/ModelOpt requires a Colab GPU runtime. Select "
            "Runtime > Change runtime type > T4 GPU, restart the runtime, "
            "and rerun the notebook from the beginning."
        )


print("GPU runtime ready:", GPU_RUNTIME_AVAILABLE)

In [31]:
# Non-TensorRT pipeline:
#  - FP32 HF model
#  - Export to ONNX
#  - ONNX runtime PTQ INT8 quantization
# - Benchmark / results

In [ ]:
import sys
from pathlib import Path

import onnx
import onnxruntime as ort
import onnxscript
import timm
import torch
import transformers

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("ONNX:", onnx.__version__)
print("ORT:", ort.__version__)
print("timm:", timm.__version__)
print("Transformers:", transformers.__version__)
print("ONNX Script:", onnxscript.__version__)
print("CUDA:", torch.cuda.is_available())
print("ORT providers:", ort.get_available_providers())

# Mount Drive only when it is not already mounted.
from google.colab import drive

DRIVE_ROOT = Path("/content/drive")
if not (DRIVE_ROOT / "MyDrive").exists():
    drive.mount(str(DRIVE_ROOT))

PROJECT = DRIVE_ROOT / "MyDrive/PTQ_MobileNet"

for folder in [
    PROJECT / "models" / "fp32_onnx",
    PROJECT / "models" / "int8_onnx",
    PROJECT / "models" / "modelopt_qdq",
    PROJECT / "models" / "trt_engine",
    PROJECT / "data" / "calibration",
    PROJECT / "data" / "evaluation",
    PROJECT / "results",
]:
    folder.mkdir(parents=True, exist_ok=True)

FP32_ONNX = PROJECT / "models/fp32_onnx/mobilenet_fp32.onnx"
INT8_ONNX = PROJECT / "models/int8_onnx/mobilenet_ort_int8.onnx"

print("Project:", PROJECT)
print("FP32 model:", FP32_ONNX)

Python: 3.13.15
Torch: 2.11.0+cu130
ONNX: 1.23.1
ORT: 1.30.0
timm: 1.0.30
Transformers: 4.57.6
ONNX Script: 0.7.2
CUDA: True
['AzureExecutionProvider', 'CPUExecutionProvider']
/content/drive/MyDrive/PTQ_MobileNet/models/fp32_onnx/mobilenet_fp32.onnx


## Export PyTorch model to ONNX

In [33]:
# Non-TensorRT pipeline work
model_name = "mobilenetv3_small_050.lamb_in1k"
model = timm.create_model(model_name, pretrained=True)
model.eval()

# Find expected model input
config = timm.data.resolve_model_data_config(model)
print(config)

# PyTorch image models expect (batch_size, channels, height, width)
dummy_input = torch.randn(1, 3, 224, 224, dtype=torch.float32)

# Export the model to ONNX
torch.onnx.export(
    model,
    (dummy_input,),
    str(FP32_ONNX),
    input_names=["image"],
    output_names=["logits"],
    opset_version=17,
    dynamo=True,
    verify=True,
)

assert FP32_ONNX.exists()
print(f"Saved successfully: {FP32_ONNX}")
print(f"Size: {FP32_ONNX.stat().st_size / 1e6:.2f} MB")

W1001 22:33:10.909000 2666 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


{'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 0.875, 'crop_mode': 'center'}
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /project/onnx/version_converter/adapters/axes_input_to_attribute.h:55: 

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
[torch.onnx] Check the ONNX model...
[torch.onnx] Check the ONNX model... ✅
[torch.onnx] Execute the model with ONNX Runtime...
[torch.onnx] Execute the model with ONNX Runtime... ✅
[torch.onnx] Verify output accuracy...
[torch.onnx] Verify output accuracy... ✅
Saved successfully: /content/drive/MyDrive/PTQ_MobileNet/models/fp32_onnx/mobilenet_fp32.onnx
Size: 0.28 MB


In [34]:
# Static Quantization: both weights and activation layers are pre-computed before quantization
# Use cases: best for image / vision models
# Pros: faster inference, higher throughput
# Cons: more setup, require calibration dataset since activation layers
# How: pass calibrated inputs to observe activation layers. Using activation layer, find X_min and X_max.
# From this calculate 2 values: Scale and Zero-Point
# These values are directly passed to calculate the Quantized Value during the quantization process without calculating dyamically


## Static Quantization pipeline for Non-TensorRT model


In [ ]:
# Static quantization pipeline
import numpy as np
import onnxruntime
from datasets import load_dataset
from onnxruntime.quantization import (
    CalibrationDataReader,
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_static,
)

FP32_ONNX = PROJECT / "models/fp32_onnx/mobilenet_fp32.onnx"
INT8_ONNX = PROJECT / "models/int8_onnx/mobilenet_ort_int8.onnx"
INT8_ONNX.parent.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "mobilenetv3_small_050.lamb_in1k"
model = timm.create_model(MODEL_NAME, pretrained=True).eval()
calib_ds = load_dataset("AI-Lab-Makerere/beans", split="train[:100]")

data_cfg = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_cfg, is_training=False)

fp32_session = onnxruntime.InferenceSession(
    str(FP32_ONNX),
    providers=["CPUExecutionProvider"],
)
input_name = fp32_session.get_inputs()[0].name
print("ONNX input:", input_name)
print("ONNX shape:", fp32_session.get_inputs()[0].shape)
print("ONNX type:", fp32_session.get_inputs()[0].type)


class MobileNetCalibrationReader(CalibrationDataReader):
    def __init__(self, dataset, input_name, transform):
        self.dataset = dataset
        self.input_name = input_name
        self.transform = transform
        self.index = 0

    def get_next(self):
        if self.index >= len(self.dataset):
            return None

        image = self.dataset[self.index]["image"].convert("RGB")
        tensor = self.transform(image).unsqueeze(0)
        self.index += 1
        return {self.input_name: tensor.numpy().astype(np.float32)}


# QOperator avoids the QDQ input-parameter issue in the current ORT runtime.
quantize_static(
    model_input=str(FP32_ONNX),
    model_output=str(INT8_ONNX),
    calibration_data_reader=MobileNetCalibrationReader(
        calib_ds,
        input_name,
        transform,
    ),
    quant_format=QuantFormat.QOperator,
    weight_type=QuantType.QUInt8,
    activation_type=QuantType.QUInt8,
    calibrate_method=CalibrationMethod.MinMax,
)

print("Saved INT8 model:", INT8_ONNX)

ONNX input: image
ONNX shape: [1, 3, 224, 224]
ONNX type: tensor(float)


ValueError: Quantization parameters are not specified for param image. In static mode quantization params for inputs and outputs of nodes to be quantized are required.

# TensorRT pipeline


In [ ]:
"""
TensorRT pipeline:
  - FP32 HF Model
  - Export to ONNX
  - TensorRT calibration + INT8 engine build
  - Benchmark / results
"""

'\nTensorRT pipeline:\n  - FP32 HF Model\n  - Export to ONNX\n  - TensorRT calibration + INT8 engine build\n  - Benchmark / results\n'

In [ ]:
from pathlib import Path
import numpy as np
import timm
from datasets import load_dataset

# Same as non-TensorRT for setup
PROJECT = Path("/content/drive/MyDrive/PTQ_MobileNet")
CALIB_NPY = PROJECT / "calibration/beans_calib_100.npy"
CALIB_NPY.parent.mkdir(parents=True, exist_ok=True)

model = timm.create_model(
    "mobilenetv3_small_050.lamb_in1k",
    pretrained=True,
).eval()

cfg = timm.data.resolve_model_data_config(model) # Determine the input size of the config
transform = timm.data.create_transform(**cfg, is_training=False)  # Transform function to scale the inputs

beans = load_dataset("AI-Lab-Makerere/beans", split="train[:100]")

calib = np.stack([
    transform(row["image"].convert("RGB")).numpy().astype(np.float32)
    for row in beans
]) # Stack each row of the beans dataset row-wise

np.save(CALIB_NPY, calib) # Save calibration dataset to CALIB_NPY path

print("Calibration shape:", calib.shape)
print("Saved:", CALIB_NPY)

Calibration shape: (100, 3, 224, 224)
Saved: /content/drive/MyDrive/PTQ_MobileNet/calibration/beans_calib_100.npy


## Fake ModelOpt INT8 QDQ




In [ ]:
# ModelOpt creates an INT8 Q/DQ ONNX graph for TensorRT.
require_gpu_runtime()

import numpy as np
from modelopt.onnx.quantization import quantize

FP32_ONNX = PROJECT / "models/fp32_onnx/mobilenet_fp32.onnx"
QDQ_ONNX = PROJECT / "models/modelopt_qdq/mobilenet_int8_qdq.onnx"
QDQ_ONNX.parent.mkdir(parents=True, exist_ok=True)

calibration_data = np.load(CALIB_NPY)

quantize(
    onnx_path=str(FP32_ONNX),
    quantize_mode="int8",
    calibration_data=calibration_data,
    output_path=str(QDQ_ONNX),
)
print("Saved QDQ ONNX:", QDQ_ONNX)

2026-10-01 22:32:15,899 - [modelopt][onnx] - INFO - Starting quantization process for model: /content/drive/MyDrive/PTQ_MobileNet/models/fp32_onnx/mobilenet_fp32.onnx


INFO:modelopt.onnx:Starting quantization process for model: /content/drive/MyDrive/PTQ_MobileNet/models/fp32_onnx/mobilenet_fp32.onnx


2026-10-01 22:32:15,901 - [modelopt][onnx] - INFO - Quantization mode: int8


INFO:modelopt.onnx:Quantization mode: int8


2026-10-01 22:32:15,902 - [modelopt][onnx] - INFO - Preprocessing the model /content/drive/MyDrive/PTQ_MobileNet/models/fp32_onnx/mobilenet_fp32.onnx


...

2026-10-01 22:32:27,859 - [modelopt][onnx] - INFO - Validating quantized model


INFO:modelopt.onnx:Validating quantized model


2026-10-01 22:32:27,868 - [modelopt][onnx] - INFO - Quantization process completed


INFO:modelopt.onnx:Quantization process completed


Saved QDQ ONNX: /content/drive/MyDrive/PTQ_MobileNet/models/modelopt_qdq/mobilenet_int8_qdq.onnx


### TensorRT Engine


In [ ]:
# Build a strongly typed TensorRT INT8 engine from the ModelOpt Q/DQ graph.
require_gpu_runtime()

import tensorrt as trt

ENGINE_PATH = PROJECT / "models/trt_engine/mobilenet_int8.engine"
ENGINE_PATH.parent.mkdir(parents=True, exist_ok=True)

logger = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(logger)
flags = 1 << int(trt.NetworkDefinitionCreationFlag.STRONGLY_TYPED)
network = builder.create_network(flags)
parser = trt.OnnxParser(network, logger)

if not parser.parse_from_file(str(QDQ_ONNX)):
    for index in range(parser.num_errors):
        print(parser.get_error(index))
    raise RuntimeError("TensorRT could not parse the Q/DQ ONNX model")

config = builder.create_builder_config()
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
serialized = builder.build_serialized_network(network, config)

if serialized is None:
    raise RuntimeError("TensorRT engine build failed")

ENGINE_PATH.write_bytes(bytes(serialized))
print("Saved TensorRT engine:", ENGINE_PATH)

KeyboardInterrupt: 

In [ ]:
# Create a held-out validation dataset.
import numpy as np
from datasets import load_dataset

val_ds = load_dataset("AI-Lab-Makerere/beans", split="test")


def make_validation_batch(dataset, transform):
    images = []
    labels = []

    for row in dataset:
        images.append(transform(row["image"].convert("RGB")).numpy().astype(np.float32))
        labels.append(row["labels"])

    return np.stack(images), np.asarray(labels, dtype=np.int64)


x_val, y_val = make_validation_batch(val_ds, transform)
print("Validation inputs:", x_val.shape)
print("Validation labels:", y_val.shape)

In [ ]:
# Test non-TensorRT pipeline
def ort_predict(model_path, inputs):
    session = ort.InferenceSession(
        str(model_path),
        providers=["CPUExecutionProvider"]
    )

    input_name = session.get_inputs()[0].name
    output = session.run(
        None, 
        {input_name: inputs}
    )[0]
    return output

# Logits
fp32_logits = ort_predict(FP32_ONNX, x_val)
int8_logits = ort_predict(INT8_ONNX, x_val)

# Predictions
fp32_predictions = np.argmax(fp32_logits, axis=1)
int8_predictions = np.argmax(int8_logits, axis=1)

# Mean
# Compare predictions against the actual y_val
fp32_accuracy = np.mean(fp32_predictions == y_val)
int8_accuracy = np.mean(int8_predictions == y_val)

print(f"FP32 ONNX accuracy: {fp32_accuracy:.4f}")
print(f"INT8 ONNX accuracy: {int8_accuracy:.4f}")
print(f"Accuracy drop: {fp32_accuracy - int8_accuracy:.4f}")


In [ ]:
# Test numerical agreement with FP32.
def compare_outputs(reference, candidate, name):
    absolute_error = np.abs(reference - candidate)
    cosine_similarity = np.sum(reference * candidate, axis=1) / (
        np.linalg.norm(reference, axis=1) *
        np.linalg.norm(candidate, axis=1) + 1e-12
    )
    reference_predictions = np.argmax(reference, axis=1)
    candidate_predictions = np.argmax(candidate, axis=1)
    prediction_agreement = np.mean(reference_predictions == candidate_predictions)

    print(f"\n{name}")
    print(f"Maximum absolute error: {absolute_error.max():.6f}")
    print(f"Mean absolute error: {absolute_error.mean():.6f}")
    print(f"Mean cosine similarity: {cosine_similarity.mean():.6f}")
    print(f"FP32 prediction agreement: {prediction_agreement:.4f}")

    return {
        "max_absolute_error": absolute_error.max(),
        "mean_absolute_error": absolute_error.mean(),
        "cosine_similarity": cosine_similarity.mean(),
        "prediction_agreement": prediction_agreement,
    }


int8_ort_comparison = compare_outputs(
    fp32_logits,
    int8_logits,
    "INT8 ONNX vs FP32 ONNX",
)

assert fp32_logits.shape == int8_logits.shape
assert np.isfinite(int8_logits).all()
assert int8_ort_comparison["prediction_agreement"] >= 0.95
assert int8_accuracy >= fp32_accuracy - 0.05

## Test TensortRT pipeline


In [ ]:
%pip install cuda-python

In [ ]:
import numpy as np
import tensorrt as trt
from cuda import cudart


def _check_cuda(error_code, operation):
    if error_code != cudart.cudaError_t.cudaSuccess:
        raise RuntimeError(f"{operation} failed: {error_code}")


def trt_predict(engine_path, inputs):
    logger = trt.Logger(trt.Logger.ERROR)

    with open(engine_path, "rb") as engine_file:
        engine_bytes = engine_file.read()

    runtime = trt.Runtime(logger)
    engine = runtime.deserialize_cuda_engine(engine_bytes)
    if engine is None:
        raise RuntimeError("TensorRT could not deserialize the engine")

    context = engine.create_execution_context()
    input_name = None
    output_name = None

    for index in range(engine.num_io_tensors):
        name = engine.get_tensor_name(index)
        mode = engine.get_tensor_mode(name)
        if mode == trt.TensorIOMode.INPUT:
            input_name = name
        else:
            output_name = name

    if input_name is None or output_name is None:
        raise RuntimeError("Expected one TensorRT input and one output tensor")

    if not context.set_input_shape(input_name, inputs.shape):
        raise RuntimeError(f"Could not set TensorRT input shape: {inputs.shape}")

    output_shape = tuple(context.get_tensor_shape(output_name))
    output_dtype = trt.nptype(engine.get_tensor_dtype(output_name))
    outputs = np.empty(output_shape, dtype=output_dtype)

    input_size = inputs.nbytes
    output_size = outputs.nbytes
    input_device = output_device = stream = None

    try:
        error, input_device = cudart.cudaMalloc(input_size)
        _check_cuda(error, "cudaMalloc(input)")
        error, output_device = cudart.cudaMalloc(output_size)
        _check_cuda(error, "cudaMalloc(output)")
        error, stream = cudart.cudaStreamCreate()
        _check_cuda(error, "cudaStreamCreate")

        context.set_tensor_address(input_name, int(input_device))
        context.set_tensor_address(output_name, int(output_device))

        error = cudart.cudaMemcpyAsync(
            input_device,
            inputs.ctypes.data,
            input_size,
            cudart.cudaMemcpyKind.cudaMemcpyHostToDevice,
            stream,
        )[0]
        _check_cuda(error, "cudaMemcpyAsync(input)")

        if not context.execute_async_v3(stream):
            raise RuntimeError("TensorRT execution failed")

        error = cudart.cudaMemcpyAsync(
            outputs.ctypes.data,
            output_device,
            output_size,
            cudart.cudaMemcpyKind.cudaMemcpyDeviceToHost,
            stream,
        )[0]
        _check_cuda(error, "cudaMemcpyAsync(output)")
        _check_cuda(cudart.cudaStreamSynchronize(stream)[0], "cudaStreamSynchronize")
        return outputs
    finally:
        if stream is not None:
            cudart.cudaStreamDestroy(stream)
        if input_device is not None:
            cudart.cudaFree(input_device)
        if output_device is not None:
            cudart.cudaFree(output_device)

In [ ]:
require_gpu_runtime()

trt_logits = trt_predict(ENGINE_PATH, x_val)
trt_predictions = np.argmax(trt_logits, axis=1)
trt_accuracy = np.mean(trt_predictions == y_val)

print(f"TensorRT INT8 accuracy: {trt_accuracy:.4f}")

trt_comparison = compare_outputs(
    fp32_logits,
    trt_logits,
    "TensorRT INT8 vs FP32 ONNX",
)

assert trt_logits.shape == fp32_logits.shape
assert np.isfinite(trt_logits).all()
assert trt_comparison["prediction_agreement"] >= 0.95
assert trt_accuracy >= fp32_accuracy - 0.05

In [ ]:
import pandas as pd

results = pd.DataFrame([
    {
        "pipeline": "FP32 ONNX Runtime",
        "accuracy": fp32_accuracy,
        "prediction_agreement": 1.0,
    },
    {
        "pipeline": "INT8 ONNX Runtime",
        "accuracy": int8_accuracy,
        "prediction_agreement": int8_ort_comparison["prediction_agreement"],
    },
    {
        "pipeline": "INT8 TensorRT",
        "accuracy": trt_accuracy,
        "prediction_agreement": trt_comparison["prediction_agreement"],
    },
])

results